In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load data
cpcr = pd.read_excel("CPCR_finalx.xls", engine="xlrd")
X = cpcr.drop("SUCCESS", axis=1).values
y = cpcr["SUCCESS"].values

# Add intercept
def add_intercept_column(X: np.ndarray) -> np.ndarray:
    n = X.shape[0]
    return np.column_stack((np.ones(n, dtype=float), X))

# Sigmoid
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-z))

# Predicted probabilities
def predict_proba(x_design: np.ndarray, beta: np.ndarray) -> np.ndarray:
    return sigmoid(x_design @ beta)

# Gradient
def compute_gradient(x_design: np.ndarray, y: np.ndarray, p: np.ndarray) -> np.ndarray:
    return x_design.T @ (y - p)

# Weight matrix
def compute_weight_matrix(p: np.ndarray) -> np.ndarray:
    return np.diag(p * (1 - p))

# Hessian
def compute_hessian(x_design: np.ndarray, w: np.ndarray) -> np.ndarray:
    return x_design.T @ w @ x_design

# IRLS fitting
def irls_fit(x_design, y, max_iter=100, tol=1e-6):
    n, d = x_design.shape
    beta = np.zeros(d)

    for _ in range(max_iter):
        p = predict_proba(x_design, beta)
        gradient = compute_gradient(x_design, y, p)
        W = compute_weight_matrix(p)
        H = compute_hessian(x_design, W)

        delta = np.linalg.pinv(H) @ gradient
        beta_new = beta + delta

        if np.linalg.norm(beta_new - beta) < tol:
            beta = beta_new
            break

        beta = beta_new

    return beta

# Predict classes
def predict_class(x_design, beta, threshold=0.5):
    p = predict_proba(x_design, beta)
    return (p >= threshold).astype(int)

# Train model
def fit_cpcr():
    x_design = add_intercept_column(X)
    beta = irls_fit(x_design, y)
    y_pred = predict_class(x_design, beta)
    acc = float(np.mean(y_pred == y))
    return beta, acc, y_pred

def main():
    beta, acc, y_pred = fit_cpcr()
    print("[CPCR] training accuracy: ", acc)

    var_names = cpcr.columns.tolist()
    var_names = [col for col in var_names if col != "SUCCESS"]
    var_names = ['Intercept'] + var_names

    coef_df = pd.DataFrame({
        'Variable': var_names,
        'Beta': beta
    })

    print(coef_df)

if __name__ == "__main__":
    main()

[CPCR] training accuracy:  0.8734177215189873
             Variable       Beta
0           Intercept  -1.259855
1              Weight   0.016432
2                MALE   0.178914
3                NEUT  -0.379821
4                 BCS  -0.100874
5   UNDER_ANAESTHESIA   2.878471
6                 DOA  -2.892400
7           Multiples  -2.363628
8                  CV  -0.846409
9        FatalArrhyth -27.448818
10        OTHER_CAUSE   1.137119
11              arrh1   0.456866
12              arrh2  -1.452248
13              arrh3  -1.998509
14              arrh4  51.400311
15              arrh5   0.594376
16              arrh6 -26.335918
17         AfterHours  -0.449225
18           Duration  -0.125435
19           Mannitol   4.732514
20               LIDO   5.632745
21             FLUIDS   1.163586
22            ABDCOMP  -0.332122
23               DOPA   4.422733
24                DOB  -2.818931
25           STEROIDS   5.997999
26               VASO   4.211558
27         ROUNDS_EPI  -0.6720

The overall fit of the logistic regression model is 87.34% training accuracy. This means that it is better than a classifier that would always predict failure, because there are 55 successes out of 158 dogs, which is roughly 35%, so the model that always predicts failure would have 65% accuracy. We can also see in the coeﬀicients that there are three extreme values, which correspond to FatalArrhyth (-27.449), arrh4 (51.40), and arrh6 (-26.336). This is showing complete or nearly complete separation, when predictor variables perfcetly or nearly predict the binary outcome, causing maximum likelihood estimates to fail. When looking at the data given, the outcome where arrh4 = 1 occurs once, arrh6 = 1 occurs twice, and FatalArrhyth = 1 occurs 15 times. It is obvious that complete separation happens for the predictors arrh4, since it is such a rare occurance in the data set that it tries to fit a coeﬀicient based on whether or not one dog died or survived. A very similar thing happens for arrh6, except with two dogs. Nearly complete separation is happening with FatalArrhyth, but it is more useful than arrh4 and arrh6. What we should do is drop arrh4 and arrh6 from the model and refit the model for better generalization and interpretability.

In [2]:
X_new_df = cpcr.drop(["SUCCESS", "arrh4", "arrh6"], axis=1)
X_new_columns = X_new_df.columns
X_new = X_new_df.values

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_new, y, test_size=0.25, random_state=0
)

scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train_lr)
X_test_lr = scaler.transform(X_test_lr)

# Fit model
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_lr, y_train_lr)

y_pred_lr = log_reg.predict(X_test_lr)

# Evaluation
print("Logistic Train Accuracy:", log_reg.score(X_train_lr, y_train_lr))
print("Logistic Test Accuracy:", log_reg.score(X_test_lr, y_test_lr))
print("Logistic F1 Score:", f1_score(y_test_lr, y_pred_lr))
print("Logistic Confusion Matrix:\n", confusion_matrix(y_test_lr, y_pred_lr))

coef_df = pd.DataFrame({
    'Variable': ['Intercept'] + list(X_new_columns),
    'Beta': np.concatenate(([log_reg.intercept_[0]], log_reg.coef_[0]))
})

print(coef_df)

Logistic Train Accuracy: 0.8983050847457628
Logistic Test Accuracy: 0.65
Logistic F1 Score: 0.5333333333333333
Logistic Confusion Matrix:
 [[18  8]
 [ 6  8]]
             Variable      Beta
0           Intercept -1.263524
1              Weight  0.111975
2                MALE  0.101621
3                NEUT -0.110250
4                 BCS -0.203562
5   UNDER_ANAESTHESIA  0.666281
6                 DOA -0.212997
7           Multiples -0.434709
8                  CV -0.294295
9        FatalArrhyth -0.514932
10        OTHER_CAUSE  0.402027
11              arrh1  0.750721
12              arrh2 -0.210392
13              arrh3 -0.503036
14              arrh5  0.264569
15         AfterHours  0.330316
16           Duration -0.658991
17           Mannitol  0.995358
18               LIDO  0.888718
19             FLUIDS  0.528458
20            ABDCOMP -0.016543
21               DOPA  0.430046
22                DOB -0.181733
23           STEROIDS  0.995236
24               VASO  0.807605
25        

With the new training accuracy being 89.83%, it did increase slightly from the first model, and it is a more stable model without arrh4 and arrh6. Even though the training accuracy is very high, it is misleading, because our data is imbalanced and logistic regression can overfit after removing variables. When looking at the coeﬀicents for the predictors, it is obvious that the predictors that will increase the odds of success the most are STEROIDS, LIDO, Lateral, Mannitol, VASO, and under anaesthesia. All of these predictors have coeﬀicients that are greater than the others. This means that dogs that have recieved steroids, lidocane, manitol, vasopressin, or were under anaesthesia as a treatment are more likely to survive. Also dogs whose body position was lateral for external compression have a higher odds of survival compared to dogs that are not lateral. The predictors that decrease the odds of success the most are rounds of Epinephrine and the duration of the attempt. This means that dogs that have more rounds of Epinephrine and have longer duration of the CPCR attempt have a decreased log-odds of success.

In [3]:
X_rf = cpcr.drop(["SUCCESS", "arrh4", "arrh6"], axis=1)
feature_names = X_rf.columns.tolist()
y = cpcr["SUCCESS"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_rf, y, test_size=0.25, random_state=0
)
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=0
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Train Accuracy:", rf.score(X_train, y_train))
print("Random Forest Test Accuracy:", rf.score(X_test, y_test))
print("F1 Score:", f1_score(y_test, y_pred_rf, average='weighted'))
cm_rf = confusion_matrix(y_test, y_pred_rf)
print("Random Forest Confusion Matrix:\n", cm_rf)
importance_df = pd.DataFrame({
    'Variable': feature_names,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\nVariable Importances:")
print(importance_df.to_string())

Random Forest Train Accuracy: 0.9067796610169492
Random Forest Test Accuracy: 0.7
F1 Score: 0.70625
Random Forest Confusion Matrix:
 [[18  8]
 [ 4 10]]

Variable Importances:
             Variable  Importance
0          ROUNDS_EPI    0.154050
1            Duration    0.139543
2              Weight    0.103697
3              FLUIDS    0.071026
4                  CV    0.065884
5             Lateral    0.063086
6          ROUNDS_ATR    0.059265
7                 BCS    0.053975
8               arrh1    0.037870
9                MALE    0.026886
10          Multiples    0.026681
11        OTHER_CAUSE    0.026287
12               DOPA    0.024075
13         AfterHours    0.019049
14              arrh5    0.017438
15  UNDER_ANAESTHESIA    0.016279
16               NEUT    0.015684
17              arrh2    0.014835
18       FatalArrhyth    0.014266
19           Mannitol    0.012454
20              arrh3    0.009815
21            ABDCOMP    0.009157
22               VASO    0.006873
23       

According to the random forest model, the most important variables are the rounds of epinephrine, the duration of the CPCR attempt, and the weight of the dog. This is very similar to the logistic regression model, because the rounds of epinephrine and the duration of the attempt decreased the log-odds of success the most. This model also scores higher than the logistic regression, but it still overfits slightly.

In [4]:
X_train_knn = scaler.fit_transform(X_train)
X_test_knn = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_knn, y_train)

y_pred_knn = knn.predict(X_test_knn)

print("KNN Train Accuracy:", knn.score(X_train_knn, y_train))
print("KNN Test Accuracy:", knn.score(X_test_knn, y_test))
print("F1 Score:", f1_score(y_test, y_pred_knn, average='weighted'))

cm_knn = confusion_matrix(y_test, y_pred_knn)
print("KNN Confusion Matrix:\n", cm_knn)

KNN Train Accuracy: 0.8813559322033898
KNN Test Accuracy: 0.75
F1 Score: 0.7288401253918495
KNN Confusion Matrix:
 [[24  2]
 [ 8  6]]


I chose the three models I did because logisitc regression focuses on the individual variables, random forest focuses on the interaction between the variables and nonlinear effects, and K-nearest neighbors is a good baseline comparison. I used these three models to predict the CPCR success and which variables are most important for this. I first implemented my own logistic regression model, and I found that the variables arrh4 and arrh6 were causing complete separation, meaning large and unstable coefficient estimates in the logistic regression model. I decided to remove these variables and use scikit learn to implement logistic regression again on the dataset. This will improve the model stability and interpretability. The original model had a training accuracy of 87.34%, and the model with the variables removed had a training accuracy of 89.83%. I then also found the test accuracy for this improved model, which was 65%. This indicates overfitting occured in the logistic regression model, and it may not be the best choice. With an F1 score of 0.53 our model has acceptable performance. I also found the confusion matrix for this model, and it suggested that this model was better at predicting failures and worse are predicting successes. This makes sense since there are not a lot of successes in the original data set. From the logistic regression model, we found that the variables that had the highest coefficients and increased the log-odds of succcess was a lateral body position for external compressions, and treating the dogs with steroids, mannitol, lidocane, and vasopressin. The variables that decreased the log-odds of success were the number of rounds of epinephrine, and the duration of the attempt. Moving on to the random forest model, we can see that there is a higher training accuracy, 90.68%, and a higher test accuracy of 70%. While this is still probably overfitting, it is better than the logistic regression model. In this model, the variables with the highest importance were the rounds of epinephrine, the duration of the attempt, and the weight of the dog. We found the F1 score to be 0.70625, which suggests a good performance of the model. In the confustion matrix for the random forest model, it suggests that it is better at predicting successes when compared to the logistic regression model. Finally, I used the KNN model, and found a training accuracy of 88.14% and a test accuracy of 75%, which has the smallest gap, which means it would overfit the least. This model also has the highest F1 score of 0.7288, which means it has a slightly better performance when compared to the random forest. The confusion matrix also shows that it is very god at predicting failures, but weaker at predicting successes. Even though the KNN model is the best performance-wise, it does not tell us which features are the most important. Therefore, I would go with the random forest model, which has good performance and good interpretability. It tells us that the most important variables for vets to focus on are the rounds of epinephrine, the duration, and the weight. I would also suggest to the vets that they should be treatinf the dogs with steroids, mannitol, lidocane, and vasopressin, since the logistic regression model said this would increase the odds of success the most.